In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
print("✅ Libraries imported!")

In [ ]:
# ── Load the new dataset ───────────────────────────────────────────────────
# Change the path to actual file location
df = pd.read_csv(r'C:\Users\WELCOME\Downloads\archive (final1)\AirQualityData.csv')# CHANGED: Combine Date + Time into a proper datetime column
# Old code only had a 'date' column; new dataset has separate Date and Time
df['Datetime'] = pd.to_datetime(
    df['Date'].astype(str) + ' ' + df['Time'].astype(str),
    errors='coerce'
)

# Drop rows where datetime parsing failed
df = df.dropna(subset=['Datetime'])

# CHANGED: Sort by Datetime (no state/area grouping in new dataset)
df = df.sort_values(by='Datetime').reset_index(drop=True)

print("✅ Data loaded and sorted by Datetime.")
print(f"   Shape: {df.shape}")
print(f"   Date range: {df['Datetime'].min()} → {df['Datetime'].max()}")
df[['Datetime', 'AirQualityIndex', 'PM2.5', 'NO2(GT)']].head()

In [ ]:
# ── 1. Lag Features ────────────────────────────────────────────────────────
# CHANGED: No groupby needed — single station dataset
# Old: df.groupby(['state', 'area'])['aqi_value'].shift(1)
# New: directly shift on the full sorted series

df['aqi_lag_1'] = df['AirQualityIndex'].shift(1)   # AQI 1 hour ago
df['aqi_lag_2'] = df['AirQualityIndex'].shift(2)   # AQI 2 hours ago

# ── 2. Rolling Window ──────────────────────────────────────────────────────
# CHANGED: No groupby, no .transform() wrapper needed
# Old: df.groupby(['state','area'])['aqi_value'].transform(lambda x: x.rolling(3).mean())
# New: rolling directly on the column

df['aqi_roll_mean_3'] = df['AirQualityIndex'].rolling(window=3).mean()

# Also add rolling means for key pollutants — new dataset has hourly data
# so a 3-hour and 24-hour rolling window are useful
df['pm25_roll_mean_3']  = df['PM2.5'].rolling(window=3).mean()
df['no2_roll_mean_3']   = df['NO2(GT)'].rolling(window=3).mean()
df['aqi_roll_mean_24']  = df['AirQualityIndex'].rolling(window=24).mean()  # daily window

# ── 3. Fill NaNs from shifting/rolling ─────────────────────────────────────
df = df.fillna(df.median(numeric_only=True))

# ── 4. Date Engineering ────────────────────────────────────────────────────
# CHANGED: New dataset already provides DayOfWeek and Hour columns
# We only add Month and Quarter (not already present)
df['month']   = df['Datetime'].dt.month
df['quarter'] = df['Datetime'].dt.quarter

# NOTE: 'day_of_week' is already in the dataset as 'DayOfWeek' — no need to recreate
# NOTE: 'hour' is already in the dataset as 'Hour' — no need to recreate

# ── 5. Label Encoding — REMOVED ────────────────────────────────────────────
# Old code encoded: state, area, prominent_pollutants
# New dataset has NO text/categorical columns — all columns are numeric
# So this entire block is removed.

print("✅ Advanced features created!")
df[['Datetime', 'AirQualityIndex', 'aqi_lag_1', 'aqi_roll_mean_3', 'aqi_roll_mean_24']].head(8)

In [ ]:
# ── Define target ──────────────────────────────────────────────────────────
# CHANGED: old target was 'aqi_value', new dataset uses 'AirQualityIndex'
TARGET = 'AirQualityIndex'

# ── Columns to drop from features ─────────────────────────────────────────
# CHANGED: different non-feature columns compared to old dataset
# Old: ['aqi_value', 'air_quality_status', 'note', 'date', 'unit']
# New: drop target + original date/time string columns (Datetime is now an index concept)
drop_cols = [
    TARGET,       # The value we are predicting — must not be a feature
    'Date',       # Raw date string — replaced by Datetime + engineered features
    'Time',       # Raw time string — replaced by Hour column
    'Datetime',   # Full datetime object — not a numeric feature
]

# Build X by dropping only columns that actually exist
X = df.drop(columns=[col for col in drop_cols if col in df.columns])
y = df[TARGET]

# UNCHANGED: same 80/20 split logic as before
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"✅ Features ready!")
print(f"   Feature count : {X.shape[1]}")
print(f"   Train samples : {len(X_train)}")
print(f"   Test  samples : {len(X_test)}")
print(f"\n   Features used:")
for i, col in enumerate(X.columns, 1):
    print(f"   {i:>2}. {col}")

In [ ]:
# ── XGBoost model — same hyperparameters as original tuned code ────────────
model = xgb.XGBRegressor(
    n_estimators=2000,        # More trees for deeper learning
    learning_rate=0.03,       # Smaller steps for better precision
    max_depth=7,              # Slightly deeper trees to catch complex patterns
    subsample=0.8,
    colsample_bytree=0.8,
    early_stopping_rounds=50, # Stop if no improvement for 50 rounds
    random_state=42
)

# UNCHANGED: same .fit() call with eval_set for early stopping
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=100               # Print progress every 100 trees
)

print("\n✅ Tuned XGBoost Training Complete!")

In [ ]:
# ── Predictions ────────────────────────────────────────────────────────────
preds = model.predict(X_test)

# ── Metrics — UNCHANGED logic, updated variable names ─────────────────────
final_r2  = r2_score(y_test, preds)
final_mae = mean_absolute_error(y_test, preds)

print("--- ACCURACY RESULTS ---")
print(f"R² Score : {final_r2:.4f}  (1.0 = perfect, >0.90 = excellent)")
print(f"MAE      : {final_mae:.2f} AQI units on average")

In [ ]:
# ── Scatter Plot: Actual vs Predicted ─────────────────────────────────────
plt.figure(figsize=(12, 6))
plt.scatter(y_test, preds, alpha=0.5, color='teal', s=15)

# Perfect prediction reference line
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    'r--', lw=2, label='Perfect Prediction'
)

# CHANGED: updated axis labels to match new dataset column name
plt.xlabel("Actual AirQualityIndex", fontsize=12)
plt.ylabel("Predicted AirQualityIndex", fontsize=12)
plt.title("Visualizing Accuracy: Actual vs Predicted AQI", fontsize=13)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature Importance — UNCHANGED ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
xgb.plot_importance(model, max_num_features=15, ax=ax, color='steelblue')
ax.set_title("Top 15 Most Important Features", fontsize=13)
plt.tight_layout()
plt.show()

# Also print as a sorted dataframe for easy reading
importance_df = pd.DataFrame({
    'Feature':    X.columns,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop 10 Features:")
print(importance_df.head(10).to_string(index=False))

In [ ]:
# ── NEW: Time-series line plot of actual vs predicted ─────────────────────
# Rebuild a results dataframe with the original Datetime index for test rows
test_indices = y_test.index
results_df = pd.DataFrame({
    'Datetime':  df.loc[test_indices, 'Datetime'].values,
    'Actual':    y_test.values,
    'Predicted': preds
}).sort_values('Datetime')

plt.figure(figsize=(14, 5))
plt.plot(results_df['Datetime'], results_df['Actual'],
         color='steelblue', linewidth=1.2, label='Actual AQI', alpha=0.85)
plt.plot(results_df['Datetime'], results_df['Predicted'],
         color='tomato', linewidth=1.2, linestyle='--', label='Predicted AQI', alpha=0.85)

plt.xlabel('Date', fontsize=11)
plt.ylabel('AirQualityIndex', fontsize=11)
plt.title('Actual vs Predicted AQI — Over Time (Test Set)', fontsize=13)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import joblib

# Save model
joblib.dump(model, 'aqi_xgboost_model.pkl')

# Save the feature column list — needed to ensure correct column order when predicting later
joblib.dump(list(X.columns), 'aqi_feature_columns.pkl')

print("✅ Model saved as: aqi_xgboost_model.pkl")
print("✅ Feature list saved as: aqi_feature_columns.pkl")
print("\nTo reload later:")
print("  model    = joblib.load('aqi_xgboost_model.pkl')")
print("  features = joblib.load('aqi_feature_columns.pkl')")